## Импорт библиотек

In [1]:
import pandas as pd
import sqlite3

## Подключение к базе

In [2]:
connection = sqlite3.connect("../data/checking-logs.sqlite")

## Обработка и анализ данных

In [3]:
pd.io.sql.read_sql(
    """
    SELECT uid, MIN(datetime) AS first_view_ts
    FROM pageviews
    WHERE uid GLOB 'user_*'
    GROUP BY uid;
    """,
    connection
)

Out[3]: 
        uid               first_view_ts
0    user_1  2020-04-26 21:53:59.624136
1   user_10  2020-04-18 12:19:50.182714
2   user_14  2020-04-18 10:53:52.623447
3   user_17  2020-04-18 10:56:55.833899
4   user_18  2020-04-26 22:49:29.243278
5   user_19  2020-04-21 20:30:38.034966
6   user_21  2020-04-22 22:40:36.824081
7   user_25  2020-05-09 23:54:54.260791
8   user_28  2020-05-10 21:07:50.350946
9    user_3  2020-05-08 10:53:47.123832
10  user_30  2020-04-17 22:46:26.785035
11   user_5  2020-05-22 11:30:18.368990
12   user_9  2020-05-21 16:28:28.217529


## Обработка и анализ данных

In [4]:
pd.io.sql.read_sql(
    """
    SELECT uid, labname, MIN(timestamp) AS first_commit_ts
    FROM checker
    WHERE status = 'ready'
      AND numTrials = 1
      AND uid GLOB 'user_*'
      AND labname IN (
          'laba04', 'laba04s', 'laba05',
          'laba06', 'laba06s', 'project1'
      )
    GROUP BY uid, labname;
    """,
    connection
)

Out[4]: 
        uid   labname             first_commit_ts
0    user_1    laba04  2020-04-26 17:06:18.462708
1    user_1   laba04s  2020-04-26 17:12:11.843671
2    user_1    laba05  2020-05-02 19:15:18.540185
3    user_1    laba06  2020-05-17 16:26:35.268534
4    user_1   laba06s  2020-05-20 12:23:37.289724
..      ...       ...                         ...
135  user_8   laba04s  2020-04-19 10:22:35.761944
136  user_8    laba05  2020-05-02 13:28:07.705193
137  user_8    laba06  2020-05-16 17:56:15.755553
138  user_8   laba06s  2020-05-16 20:01:07.900727
139  user_8  project1  2020-05-14 15:42:04.002981

[140 rows x 3 columns]


## Обработка и анализ данных

In [5]:
connection.execute("DROP TABLE IF EXISTS test;")
connection.execute("DROP TABLE IF EXISTS control;")
connection.execute("DROP TABLE IF EXISTS datamart;")
connection.commit()

## Создание витрины данных

In [6]:
connection.execute(
    """
    CREATE TABLE datamart AS
    SELECT
        c.uid,
        c.labname,
        c.first_commit_ts,
        p.first_view_ts
    FROM (
        SELECT uid, labname, MIN(timestamp) AS first_commit_ts
        FROM checker
        WHERE status = 'ready'
          AND numTrials = 1
          AND uid GLOB 'user_*'
          AND labname IN (
              'laba04', 'laba04s', 'laba05',
              'laba06', 'laba06s', 'project1'
          )
        GROUP BY uid, labname
    ) AS c
    LEFT JOIN (
        SELECT uid, MIN(datetime) AS first_view_ts
        FROM pageviews
        WHERE uid GLOB 'user_*'
        GROUP BY uid
    ) AS p
        ON c.uid = p.uid;
    """
)

connection.commit()

## Обработка и анализ данных

In [7]:
datamart = pd.io.sql.read_sql(
    "SELECT * FROM datamart;",
    connection,
    parse_dates=["first_commit_ts", "first_view_ts"]
)

datamart = datamart.astype({
    "first_commit_ts": "datetime64[ns]",
    "first_view_ts": "datetime64[ns]"
})

datamart.head()

Out[7]: 
      uid  labname            first_commit_ts              first_view_ts
0  user_1   laba04 2020-04-26 17:06:18.462708 2020-04-26 21:53:59.624136
1  user_1  laba04s 2020-04-26 17:12:11.843671 2020-04-26 21:53:59.624136
2  user_1   laba05 2020-05-02 19:15:18.540185 2020-04-26 21:53:59.624136
3  user_1   laba06 2020-05-17 16:26:35.268534 2020-04-26 21:53:59.624136
4  user_1  laba06s 2020-05-20 12:23:37.289724 2020-04-26 21:53:59.624136


## Обработка и анализ данных

In [8]:
pd.io.sql.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table';",
    connection
)

Out[8]: 
        name
0  pageviews
1    checker
2  deadlines
3   datamart


## Обработка и анализ данных

In [9]:
test = datamart[datamart["first_view_ts"].notna()].copy()
control = datamart[datamart["first_view_ts"].isna()].copy()

## Обработка и анализ данных

In [10]:
mean_first_view = (
    test[["uid", "first_view_ts"]]
    .drop_duplicates(subset="uid")
    ["first_view_ts"]
    .mean()
)

control["first_view_ts"] = control["first_view_ts"].fillna(
    mean_first_view
)

## Обработка и анализ данных

In [11]:
print("Строк в test:", len(test))
print("Строк в control:", len(control))
print("Средняя дата:", mean_first_view)
print("Осталось пропусков:", control["first_view_ts"].isna().sum())

Строк в test: 59
Строк в control: 81
Средняя дата: 2020-04-26 11:42:34.626102272
Осталось пропусков: 0


## Сохранение аналитических групп

In [12]:
test.to_sql("test", connection, index=False)
control.to_sql("control", connection, index=False)

Out[12]: 81


## Обработка и анализ данных

In [13]:
pd.io.sql.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table';",
    connection
)

Out[13]: 
        name
0  pageviews
1    checker
2  deadlines
3   datamart
4       test
5    control


## Закрытие соединения

In [14]:
connection.close()